In [ ]:
import numpy as np
import pandas as pd
import keras 
import plotly.express as px
from allensdk.core.cell_types_cache import CellTypesCache 

Matplotlib is building the font cache; this may take a moment.


In [4]:
ctc = CellTypesCache(manifest_file="cell_types/manifest.json") # Connects to Allen Institute and saves the downloaded file  in a cell_types/ folder
df = pd.DataFrame(ctc.get_ephys_features()) # get_ephys_features() -> returns one row of measurements per neuron. Putting that into a dataframe
print(df.shape) # df.shape tells us how many neurons (rows) and measurements (columns) we have 
print(df.columns.tolist()) # columns.tolist() list all measurement names -> Will help us to confirm rheobase and resistence column names before we use them

(2333, 56)
['adaptation', 'avg_isi', 'electrode_0_pa', 'f_i_curve_slope', 'fast_trough_t_long_square', 'fast_trough_t_ramp', 'fast_trough_t_short_square', 'fast_trough_v_long_square', 'fast_trough_v_ramp', 'fast_trough_v_short_square', 'has_burst', 'has_delay', 'has_pause', 'id', 'input_resistance_mohm', 'latency', 'peak_t_long_square', 'peak_t_ramp', 'peak_t_short_square', 'peak_v_long_square', 'peak_v_ramp', 'peak_v_short_square', 'rheobase_sweep_id', 'rheobase_sweep_number', 'ri', 'sag', 'seal_gohm', 'slow_trough_t_long_square', 'slow_trough_t_ramp', 'slow_trough_t_short_square', 'slow_trough_v_long_square', 'slow_trough_v_ramp', 'slow_trough_v_short_square', 'specimen_id', 'tau', 'threshold_i_long_square', 'threshold_i_ramp', 'threshold_i_short_square', 'threshold_t_long_square', 'threshold_t_ramp', 'threshold_t_short_square', 'threshold_v_long_square', 'threshold_v_ramp', 'threshold_v_short_square', 'thumbnail_sweep_id', 'trough_t_long_square', 'trough_t_ramp', 'trough_t_short_squ

In [ ]:
# Creating a new dataframe and adding the columns from df into it.
data = df[["input_resistance_mohm", "tau", "vrest", "sag", "threshold_i_long_square"]].copy() # -> copy() just copies the 4 columns we need into a new data frame
data.columns = ["resistance", "tau", "vrest", "sag", "rheobase"] # Adds
data.head()

,resistance,tau,vrest,sag,rheobase
0,54.894264,20.495034,-73.553391,0.116452,290.0
1,103.684656,25.131172,-73.056595,0.206351,310.0
2,224.580336,18.314883,-60.277321,0.156424,30.0
3,151.193344,7.616164,-75.205559,0.041767,270.0
4,171.530176,34.829629,-63.474991,-0.003465,90.0


In [10]:
data.describe() # Running a series of descriptive staistics on the dataset 
# Data is Right-skewed. Essentially mean is being dragged upwards by a high max. 
# Therefore, we will log transform the graph 
# rheobase is inversly proportional to resistence which is a curve in itself when plotted. So taking the log of both sides turns it into a straight line that we can fit with traditional linear regression

,resistance,tau,vrest,sag,rheobase
count,2333.000000,2333.000000,2333.000000,2333.000000,2333.000000
mean,207.305669,19.247932,-71.465922,0.078844,141.018002
std,121.587165,11.035618,5.459283,0.065862,108.311157
min,23.177554,3.472114,-85.449173,-0.034598,10.000000
25%,133.767984,11.173358,-75.396614,0.031177,70.000000
50%,184.994368,17.658727,-71.975128,0.059374,110.000000
75%,249.672928,24.672806,-68.011543,0.108459,190.000000
max,2425.824000,176.893484,-54.961067,0.490941,800.000000


In [18]:
(data["rheobase"].round() % 10 == 0).mean() # Rounds the values before running a mean function on them
# Using the remainder operator. Will equal to True when no remainders
# .mean() gives the fraction of True values 1.0 means every neuron is on a 10 pA step. 

0.9995713673381912

In [ ]:
# To check the other ones with a ronded value
off = data.loc[data["rheobase"].round() % 10 != 0, "rheobase"] # Keeping all the cells with a remainder.
print(len(off)) 
print(off.value_counts().head(10)) # Shows which value appears most often

1
15.000001    1
Name: rheobase, dtype: int64


In [25]:
data.corr()["rheobase"].sort_values() # Just looking at the rheobase column. 
# Correlation matrix. -> How one colum moves with every other column, and returns the results as a table. 
# .sort_values() orders them from most negative to most positive. 
# Will default to Pearson correlation 
# Not the best measure for correlation since this is a curve 


resistance   -0.479824
tau          -0.428175
vrest        -0.356892
sag          -0.132468
rheobase      1.000000
Name: rheobase, dtype: float64

In [27]:
# Making a correlation matrix for the log data instead

logdata = np.log10(data[["resistance", "tau", "rheobase"]])
logdata.corr()["rheobase"].sort_values() # Now producing a correlation matrix for the log data 

resistance   -0.636687
tau          -0.538503
rheobase      1.000000
Name: rheobase, dtype: float64

In [ ]:
# R Squared values to measure how much rheobase explains the variation 
logdata.corr()["rheobase"] ** 2 # To measure R^2 across the columns against rheobase. So 40 percent of the variation in rheobase comes from resistence.

resistance    0.405371
tau           0.289985
rheobase      1.000000
Name: rheobase, dtype: float64

In [ ]:
px.scatter(data, x="resistance", y="rheobase", opacity=0.4, title="Raw: resistance vs rheobase")
# Plotting Raw scatter plot of rheobase against resistence and we see a curve.
# This is because of ohms law. rheobase is inversly proportional to resistence. Straight line can't follow that curve well. Loging both axis makes the resistence curve into a straight line 
# log(rheobase) = -1 * log(resistence) + b 
# before it was rheobase = 1/resistence

In [ ]:
px.scatter(logdata, x="resistance", y="rheobase", opacity=0.4, title="Raw: resistance vs rheobase")
# Another reason we made it log is because the data is skewed rightwards. Big gap between 75 % and the max means few extreme cells put the line toward themselves. 
# Log allows for bigger values more than smaller ones so the cells stop dominating

In [62]:
def build_model (learning_rate, n_features):
    inputs = keras.Input(shape=(n_features,)) # shape=(nfe...) How many columns are going in 
    outputs = keras.layers.Dense(units=1)(inputs) # Dense (units=1) -> How a single neuron computes weight * x * bias, that's the equation of the straight line. Basically the linear regression 
    model =keras.Model(inputs=inputs, outputs=outputs) 
    model.compile(
        optomizer=keras.optomizers.RMSprop(learning_rate=learning_rate), # Changes to the weights and biases after each batch | Graidient descent -> 
        # memory = 0.9 × memory + 0.1 × gradient²
        # new weight = weight − learning_rate × gradient / √memory
        loss="mean_squared_error", # What gets minimised, the average of (prediction - truth)^2 
        metrics=[keras.metrix.RootMeanSquaredError()], # square root of the loss
    )
    return model

In [ ]:
# Wrote my own training run without a machine learning library at all

x_all = logdata[["resistance"]].values # .values turns the table into a plain array that Keras accepts
y_all = logdata[["rheobase"]].values

w, b = 0.0, 0.0 # knobs start at 0

mem_w, mem_b = 0.0, 0.0 # Memory of how big the weights and bias's gradients have been recently

lr = 0.01 # Learning Rate 
rho = 0.9 # Memory decay -> How much of the past memory is kept in each step
eps = 1e-7 # Epsilon -> Tiny number that prevents dividing by 0

batch_size = 50
epochs = 50
log = [] # Empty list -> Adds one entry after every loop. Adds the bundle to the end of the list.

rng = np.random.default_rng(0) # Random number generator -> Shuffling the neurons at the start of every epoch | Used the same seed for reproducibility

for epoch in range(epochs): # 5 Passes as in 5 epochs 
    order = rng.permutation(len(x_all)) # Shuffling once per epoch, so neurons are shuffled 
    for start in range (0, len(x_all),batch_size): # Once per batch
        
        # Grab the batch
        idx=order[start:start + batch_size] # Slices the next 50 of them into idx 
        x, y = x_all[idx], y_all[idx] # x_all[idx] picks those 50 neurons' log resistance, and y_alll[idx] picks their log rheobase. 
                                      # x and y therefore are each arrays of 50 numbers

        err = (w * x +b) - y # Lines prediction for all 50 neurons ar once. NumPy applies it to every element # error / neuron
                             # Subtracting y gives us 50 errors.  
                             # This is the prediction error. So expected outcome based on the input, weight and bias. 
                             # This is the difference between what the model predicted and what actually happened. 

        # Slopes of the LOSS function for each knob -> (wx + b - y)**2                     
        grad_w = 2 * np.mean(err*x) # Multiplies error with x because weight's effect on the prediction scales with x 
        grad_b = 2 * np.mean(err) # No multiplication because b sgufts every prediction equally
                                  # if the err is -2.1, therefore grad_w = 2*(-2.1*2.3) = -9.7 -> Therefore it's basically saying. Both knobs turn up.  


        # Update the memory for the weights
        mem_w = rho*mem_w + (1-rho) * grad_w**2 # multiply the memory decay with memory of the weight and add 1-rho * the gradient squared -> This is always positive and measures size
        mem_b = rho*mem_b + (1-rho) * grad_b**2

        # Steps are taken iteratively 
        w -= lr *grad_w/(np.sqrt(mem_w) + eps) #subtract from yourself. We also divide the gradient by sqrt(memory)
        b -= lr *grad_b/(np.sqrt(mem_b) + eps)
        
        
        #Example, first step:
        #sqrt(9.4) = 3.07, so 9.7 / 3.07 = 3.16, times 0.01 = 0.0316
        #w = 0 - 0.01 * (-9.7 / 3.07) = +0.032
        #sqrt1.8 = 1.34, so b = 0 - 0.01 * (-4.2 / 1.34) = +0.031

        # Snapshot recorded 
        log.append(dict(epoch=epoch, w=w, b=b, grad_w=grad_w, mem_w=mem_w, step_w=lr * grad_w/(np.sqrt(mem_w)+eps), loss=np.mean(err**2)))
        # loss function is calculated at the end. Mean squared error for this batch specifcally. 
        # Loss is computed from err before the update and shows how wrong the line was going into step

trace =pd.DataFrame(log)
trace.head(5)


,epoch,w,b,grad_w,mem_w,step_w,loss
0,0,0.031623,0.031623,-9.111095,8.301206,-0.031623,4.575350
1,0,0.053973,0.053672,-8.635169,14.927699,-0.022350,3.888497
2,0,0.072283,0.071681,-8.231648,20.210932,-0.018310,3.473158
3,0,0.088487,0.087523,-8.048125,24.667071,-0.016205,3.245131
4,0,0.103146,0.102243,-7.794779,28.276222,-0.014659,3.209467


In [64]:
trace.tail(5) # Look at the tail of the trace 

,epoch,w,b,grad_w,mem_w,step_w,loss
935,19,-0.388406,2.904622,0.628137,0.065627,0.024520,0.092157
936,19,-0.386742,2.909341,-0.040508,0.059228,-0.001664,0.095241
937,19,-0.394368,2.905185,0.181420,0.056597,0.007626,0.079147
938,19,-0.417084,2.884184,0.736994,0.105253,0.022717,0.104717
939,19,-0.408476,2.895905,-0.275342,0.102309,-0.008608,0.098011


In [65]:
# Plotting the changes over time 
for col in ["loss", "grad_w", "mem_w", "w"]:
    px.line(trace, y=col, title=col).show()

In [59]:
# How good is the fit? 
np.polyfit(logdata["resistance"], logdata["rheobase"], 1)

array([-0.91222909,  4.08865655])

In [66]:
def build_model(learning_rate, n_features):
    inputs = keras.Input(shape=(n_features,))
    outputs = keras.layers.Dense(units=1)(inputs)
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.RMSprop(learning_rate=learning_rate),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()],
    )
    return model

In [73]:
X = logdata[["resistance"]].values
y = logdata["rheobase"].values

model = build_model(learning_rate=0.01, n_features=1)
history = model.fit(X, y, batch_size=50, epochs=100)
print(model.get_weights())

Epoch 1/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 750us/step - loss: 2.5950 - root_mean_squared_error: 1.6109
Epoch 2/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 890us/step - loss: 0.3522 - root_mean_squared_error: 0.5935
Epoch 3/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 691us/step - loss: 0.1790 - root_mean_squared_error: 0.4231
Epoch 4/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 630us/step - loss: 0.1707 - root_mean_squared_error: 0.4132
Epoch 5/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 622us/step - loss: 0.1609 - root_mean_squared_error: 0.4011
Epoch 6/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 644us/step - loss: 0.1513 - root_mean_squared_error: 0.3890
Epoch 7/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 607us/step - loss: 0.1423 - root_mean_squared_error: 0.3773
Epoch 8/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 606us/step - loss: 0.1348 - root_mean_squared_error: 0.3672
Epoch 9/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 650us/step - loss: 0.1283 - root_mean_squared_error: 0.3582
Epoch 10/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 617us/step - loss: 0.1217 - root_mean_s

In [ ]:
slope, intercept = np.polyfit(logdata["resistance"], logdata["rheobase"], 1) # Compute the exact best line(x,y,degree) -> fits a polynomial to the points 
pred = slope*logdata["resistance"]+ intercept # predict with that line. 
best_rmse = np.sqrt(np.mean((pred-logdata["rheobase"])**2)) # measure the error 
print(best_rmse) # show the results

0.2561968763190421


In [72]:
fig = px.line(y=history.history["root_mean_squared_error"],
              labels={"x": "epoch", "y": "RMSE"}, title="Keras training curve")
fig.add_hline(y=best_rmse, line_dash="dash", annotation_text="best possible")
fig.show()